# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()

In [4]:
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "llama3.2:1b"

llm = OpenAI(base_url=BASE_URL, api_key=API_KEY);

In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about'},
  {'type': 'career page', 'url': 'https://edwarddonner.com/careers'},
  {'type': 'posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:1b
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about'},
  {'type': 'proficient', 'url': 'https://edwarddonner.com/proficient'},
  {'type': 'connect-four', 'url': 'https://edwarddonner.com/connect-four'},
  {'type': 'outsmart', 'url': 'https://edwarddonner.com/outsmart'},
  {'type': 'posts', 'url': 'https://edwarddonner.com/posts'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


{'links': [{'type': 'inference/models',
   'url': 'https://huggingface.co/models'},
  {'type': 'inference/models',
   'url': 'https://huggingface.co/inference/models'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash
Updated
5 days ago
•
326k
•
2.66k
Edge0/Edge0-35B-A3B-preview
Updated
1 day ago
•
17.9k
•
2.59k
openbmb/MiniCPM5-2B
Updated
4 days ago
•
272k
•
1.44k
nex-agi/Nex-N2.5-mini
Updated
7 days ago
•
5.2k
•
803
Qwen/Qwen3.8-27B
Updated
Aug 14
•
7.7M
•
15.2k
Br

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdated\n5 days ago\n•\n326k\n•\n2.66k\nEdge0/Edge0-35B-A3B-preview\nUpdated\n1 day

In [21]:
def create_brochure(company_name, url):
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 5 relevant links


# Hugging Face: The AI Community Building the Future

## Introduction
Hugging Face is a leading platform for machine learning and artificial intelligence, where the machine learning community comes together to build, share, and discover new models, datasets, and applications.

## About Us

*   Hugging Face is a subsidiary of Meta AI.
*   The company was founded in 2014 as a private company, but was acquired by Meta AI in 2020.
*   The platform is available for free and offers a range of features, including models, datasets, and applications.
*   Hugging Face has gained significant traction in the machine learning community, with estimates suggesting that over 2 million models are hosted on the platform.

## Models

*   Hugging Face offers a wide range of pre-trained models, including those for computer vision, natural language processing, and more.
*   The platform provides a variety of models and datasets, including those from popular sources such as Stanford NLP, Amazon Rekognition, and Microsoft Azure Machine Learning.
*   Users can access and use models and datasets on their own, or use the platform's collaboration features to host and share their own models.

## Datasets

*   Hugging Face provides a wide range of public datasets, including those for computer vision, natural language processing, and more.
*   The platform also offers a range of custom datasets that can be created and hosted on the platform.
*   Datasets are free to use, and can be accessed and shared with others on the platform.

## Spaces

*   Hugging Face has created two main spaces: Research and Production.
*   Research space allows developers to share and collaborate on public models, datasets, and applications.
*   Production space allows high-quality models, datasets, and applications to be created and shared.

## Careers

*   Hugging Face has a growing team with a diverse range of engineers, researchers, and analysts.
*   The company offers a range of job opportunities, including full-time and part-time positions in research, engineering, and other fields.
*   Opportunities may be available for those with experience in machine learning, computer vision, natural language processing, or other relevant areas.

## Team & Enterprise

*   The Hugging Face team is led by CEO and CEO of Meta AI.
*   The company offers a range of benefits and perks, including flexible work arrangements, free amenities, and a comprehensive training program.
*   Enterprise access to Hugging Face PRO, Enterprise Support, Inference Providers, and Inference Endpoints is available for those who need a high-performance platform for their production workloads.

## Hugging Face PRO

*   Hugging Face PRO offers enterprise-level support for high-performance computing on top of Hugging Face.
*   The platform provides Enterprise Support, which includes dedicated support engineers, technical success teams, and advanced documentation.
*   Users can also access Hugging Face Pro Inference Provider tiers to handle large-scale production workloads.

## Inferences Providers & Inference Endpoints

*   Hugging Face offers three Inference Provider tiers:
+   Edge Zero: Free tier that allows usage of models and datasets on Edge zero, a high-performance computing platform. Tier 2 offer provides the Edge Zero platform and provides support to increase the use of Hugging Face models. Tier 1 provide all the feature and support.
+   Edge Zero Tier 2: Tier 2 provides up to 2,700 TFLOPS at 1.3 TFLOPS per model and requires 100 GB/s of I/O bandwidth, and includes support for high-density deployment and high-performance computing on large-scale Hugging Face applications.
+   Edge Zero Tier 3: Tier 3 provides all the features and support plus 5,840 TFLOPS and includes Tier 1 features.

## Pricing
The pricing for Hugging Face can be paid monthly or yearly.

Hugging Face platform and services include pricing plans for customers or enterprise customers including features for data sharing, and for users accessing HF models, inference, on large-scale production workloads which are offered across multiple pricing tiers.

## Workshops and Training
To learn more about Hugging Face, developers can attend Hugging Face workshops and training sessions. The platform offers online and offline training sessions, as well as a comprehensive workshop schedule.

## Disclosures
Hugging Face is proud to be a leading platform for machine learning and artificial intelligence. We use our platform to build, share, and discover new models, datasets, and applications. We offer a wide range of features, including models, datasets, and applications.

**Disclaimer:** This brochure is intended to provide a general overview of Hugging Face and does not include a comprehensive list of users or users' models. By utilizing Hugging Face platform and services, or their data services, you're committing to following [your company's data protection policy](https://www.huggingface.co/terms-of-usage), [Hugging Face's data protection policy](https://huggingface.co/data-protection).

If you need more detail, please contact our customer support at [support@huggingface.co](mailto:support@huggingface.co) or visit our documentation section at [docs.huggingface.co](https://docs.huggingface.co).

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


# Hugging Face: The AI Community Building the Future

## About Us

Hugging Face is the AI community that empowers users to build, share, and deploy machine learning models. Founded in [Year] with a strong focus on open-source and collaborative development, Hugging Face has become a leader in the AI community.

### Follow Xay

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
About
Careers
Website
Models
Datasets
Spaces
Pricing
Docs

## Enterprise Plans

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up

### Xay About Us

Our mission is to build a community of learners, researchers, and practitioners of machine learning. We provide a collaborative platform for developers, researchers, and analysts to build, share, and deploy machine learning models.

### Enterprise Support

Hugging Face Enterprise provides various support resources for businesses and organizations to help them integrate machine learning into their workflows. This includes support for enterprise-level features, custom development, and professional training.

### Careers

Hugging Face is committed to creating a diverse and inclusive workforce. We encourage talented individuals to apply for our various entry-level positions across different function areas.

## Models

Hugging Face provides over 2 million models across several categories, including:

* Natural Language Processing (NLP)
* Computer Vision
* Time Series Forecasting
* Text Classification
* Regression
* Clustering
* and many more

### Featured Models

Our featured models are a collection of state-of-the-art models that have achieved high accuracy and performance. Some examples include:

* AuK: A unified speech generation and editing model with state-of-the-art performance.
* Qwen: An image editing and generation model with fast rendering times and high-quality output.

## Datasets

Hugging Face provides access to over 500k+ datasets, including:

* General-purpose datasets for various applications
* Specialized datasets for specific industries and domains
* Datasets for natural language processing, computer vision, and time series forecasting

### Browse Datasets

Our datasets have been used in a wide range of applications, from natural language processing tasks to computer vision benchmarks.

## Spaces

Hugging Face provides a platform for developers to host and deploy their own models, including:

* Deploying models to production environments
* Sharing models with others and collaborating on datasets
* Setting up custom authentication and authorization

### Spaces: Collaborative Model Hosting

Our customizable hosting environment allows developers to host their own models on Hugging Face. This enables users to deploy their models to production environments quickly and easily.

## Inference Providers

Hugging Face provides a range of inference endpoints, including:

* Static inference: Inference provided from pre-trained models
* Dynamic inference: Inference provided from deployed models
* Query-by-example (QBE) inference: Inference provided from user-provided input data

### Inference Endpoints

Our inference endpoints provide developers with the ability to deploy models to production environments and use their outputs in various applications.

## Storage Buckets

Hugging Face provides a range of storage buckets, including:

* Public buckets for shared datasets
* Private buckets for secure and private datasets
* Custom buckets for users to create their own storage buckets

### Storage Buckets: Secure Data Storage

Our storage buckets provide a secure way to store and manage data for a variety of use cases, including applications, research, and analytics.

## Log In/Sign Up

You can log in or sign up for an account on our platform to access a wide range of features and resources.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


## Hugging Face - The AI Community Building the Future

### About

Hugging Face is a leading provider of cutting-edge AI models, datasets, and applications. Our mission is to empower individuals and organizations to collaborate on machine learning projects and push the boundaries of what is possible. With a strong focus on innovation, community, and inclusivity, we've created a collaborative platform for the AI community to come together.

### Models

We host a vast collection of models, including but not limited to:

* **DeeSeq**: A model for identifying differentially expressed genes in tumor samples.
* **Qwen**: A model for image editing and generation.
* **QIE**: A model for rapid AI object detection and segmentation.
* **MiniCPM**: A model for converting prices to capacity measures (MCR).

You can explore all our models on our **Models** page.

### Datasets

Our **Datasets** page showcases a wealth of publicly available datasets, including:

* **NYU ML**: A large-scale dataset for benchmarking machine learning models.
* **Squad**: A dataset for evaluating conversational AI models.
* **UltraData**: A dataset for benchmarking AI models for image classification and segmentation.

### Spaces

Our **Spaces** page highlights some of our most exciting projects, including:

* **Nyu-MLL**: A model for predicting protein structures.
* **Stanford-NLP**: A dataset for evaluating natural language processing models.

### Pricing

Our **Pricing** page provides information on our pricing plans, including a free tier and custom pricing for enterprise clients.

### Website

Our **Website** page features a user-friendly interface to access our platform, including a **Tasks** page for collaboration, a **Learn** page for tutorials and documentation, and a **Community** page for discussion forums.

### Enterprise Support

For enterprise clients, we offer **Enterprise Support**, which includes priority access to our platform, dedicated support, and customized pricing.

### Inference Endpoints

Our **Inference Endpoints** page provides a centralized place to manage and deploy models on our platform, including a **Catalog** page for managing inference providers and a **Deploy** page for deploying models in production.

### Storage Buckets

Our **Storage Buckets** page allows you to store and manage your datasets, including a **Storage** page for uploading files and a **Log In** page for login.

### Logs

Our **Logs** page provides real-time logs for all platform activity, including **Tasks**, **Spaces**, and **Models**.

### Help & Contact

For assistance or feedback, please contact our team using the **Discord** or **Forum** channels.

### Careers

For more information on how to join our team, please visit our **Careers** page.

### Solutions

Our **Solutions** page highlights additional tools and services that complement our platform, including a **Team & Enterprise** page for collaborative projects.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>